[![cloudevel](img/cloudevel.png)](https://cloudevel.com)

# Seguridad básica.

La seguridad en GNU/Linux es un tema amplio que merece un curso propio. Sin embargo, existen conceptos y herramientas fundamentales que todo usuario del sistema debe conocer, independientemente de su rol.

Este capítulo cubre los fundamentos que se aplican en el uso cotidiano del sistema. Para un tratamiento completo de administración de seguridad, cifrado, auditoría avanzada y control de acceso obligatorio, consulte el curso de seguridad en sistemas GNU/Linux.

## El modelo de seguridad de GNU/Linux.

GNU/Linux hereda el modelo de seguridad de *UNIX*, basado en tres principios:

* **Identidad**: cada proceso y cada archivo pertenece a un usuario y a un grupo.
* **Permisos discrecionales**: el propietario de un recurso decide quién puede acceder a él.
* **Privilegios mínimos**: cada proceso debe ejecutarse con los permisos estrictamente necesarios para su función.

El tercer principio es el más frecuentemente ignorado y el que causa más incidentes de seguridad.

## Por qué no trabajar como ```root```.

El usuario ```root``` tiene acceso irrestricto a todos los recursos del sistema. Trabajar habitualmente como ```root``` implica que cualquier error tipográfico, script malicioso o programa comprometido opera también con esos privilegios.

La práctica correcta es:

1. Crear un usuario ordinario para el trabajo cotidiano.
2. Otorgar a ese usuario acceso a ```sudo``` para las tareas que lo requieran.
3. Reservar el acceso directo a ```root``` para emergencias de recuperación del sistema.

El comando ```whoami``` permite verificar con qué usuario se está operando en un momento dado.

In [ ]:
whoami

In [ ]:
id

## Contraseñas.

### El archivo ```/etc/shadow```.

Las contraseñas de los usuarios no se almacenan en texto plano. El archivo ```/etc/shadow``` contiene los *hashes* de las contraseñas y solo es legible por ```root```.

Cada línea de ```/etc/shadow``` contiene, entre otros campos:

* El nombre de usuario.
* El hash de la contraseña (algoritmo, sal y valor).
* La fecha del último cambio de contraseña.
* Políticas de expiración.

Un hash que comienza con ```$6$``` indica SHA-512; ```$y$``` indica yescrypt, el algoritmo preferido en sistemas modernos.

In [ ]:
sudo cat /etc/shadow | head -5

### El comando ```passwd```.

Permite cambiar la contraseña de un usuario.

```
passwd            # cambia la contraseña del usuario actual
sudo passwd <usuario>   # cambia la contraseña de otro usuario (requiere root)
```

**Buena práctica:** use contraseñas largas (más de 16 caracteres) o frases de paso. La longitud es más importante que la complejidad.

## Permisos especiales.

Además de los permisos de lectura, escritura y ejecución, existen tres bits especiales que modifican el comportamiento de archivos y directorios.

### ```setuid``` (SUID).

Cuando se activa en un archivo ejecutable, el proceso se ejecuta con la identidad del **propietario del archivo**, no del usuario que lo invoca.

El ejemplo clásico es ```passwd```: cualquier usuario puede cambiar su propia contraseña porque el ejecutable tiene SUID de ```root```, lo que le permite escribir en ```/etc/shadow```.

In [ ]:
ls -l /usr/bin/passwd

La ```s``` en la posición del bit de ejecución del propietario indica SUID activo.

**Advertencia:** los archivos con SUID de ```root``` son puntos de atención prioritaria en una auditoría de seguridad. La siguiente celda lista todos los que existen en el sistema.

In [ ]:
find / -perm -4000 -type f 2>/dev/null

### ```setgid``` (SGID).

En un archivo ejecutable, el proceso corre con la identidad del **grupo propietario** del archivo.

En un directorio, los archivos creados dentro heredan el grupo del directorio en lugar del grupo primario del usuario que los crea. Es útil para directorios de trabajo colaborativo.

### *Sticky bit*.

En un directorio, impide que un usuario borre o renombre archivos que no le pertenecen, aunque tenga permiso de escritura en el directorio.

El caso más conocido es ```/tmp```.

In [ ]:
ls -ld /tmp

La ```t``` al final de los permisos indica *sticky bit* activo.

## ```umask```.

El ```umask``` es una máscara que determina los permisos que se **niegan por omisión** al crear nuevos archivos y directorios.

El valor más común es ```0022```, que produce:

* Archivos nuevos con permisos ```644``` (rw-r--r--).
* Directorios nuevos con permisos ```755``` (rwxr-xr-x).

In [ ]:
umask

Un ```umask``` de ```0027``` es más restrictivo: niega todo acceso a usuarios fuera del grupo propietario. Es el valor recomendado en entornos multi-usuario sensibles.

## Auditoría básica.

### Sesiones de usuarios.

Los siguientes comandos permiten revisar quién ha accedido al sistema y cuándo.

In [ ]:
last | head -20

In [ ]:
lastfail 2>/dev/null || last -f /var/log/btmp 2>/dev/null | head -10

El archivo ```/var/log/auth.log``` (en sistemas Debian/Ubuntu) o ```/var/log/secure``` (en sistemas Red Hat) registra todos los eventos de autenticación: accesos exitosos, fallidos, uso de ```sudo``` y cambios de contraseña.

In [ ]:
sudo tail -20 /var/log/auth.log 2>/dev/null || sudo journalctl -u ssh --no-pager | tail -20

## Buenas prácticas con SSH.

SSH es el protocolo estándar para acceso remoto en GNU/Linux. Su configuración tiene un impacto directo en la seguridad del sistema.

### Autenticación con llaves.

La autenticación por contraseña es vulnerable a ataques de fuerza bruta. La autenticación con par de llaves (pública/privada) es significativamente más segura.

Para generar un par de llaves:

In [ ]:
# Genera un par de llaves Ed25519 (algoritmo recomendado)
# ssh-keygen -t ed25519 -C "comentario identificador"
echo "El comando anterior requiere interacción. Ejecútelo en una terminal."

La llave pública (```~/.ssh/id_ed25519.pub```) se deposita en el servidor en ```~/.ssh/authorized_keys```. La llave privada nunca debe salir del equipo del usuario.

### Configuración recomendada del servidor SSH.

El archivo de configuración del servidor es ```/etc/ssh/sshd_config```. Las siguientes directivas mejoran significativamente la seguridad:

```
PermitRootLogin no          # Deshabilita el acceso directo como root
PasswordAuthentication no   # Fuerza el uso de llaves
MaxAuthTries 3              # Limita intentos por conexión
```

**Nota:** antes de deshabilitar la autenticación por contraseña, verifique que su llave funciona correctamente. De lo contrario quedará bloqueado fuera del sistema.

In [ ]:
sudo grep -E "^PermitRootLogin|^PasswordAuthentication|^MaxAuthTries" /etc/ssh/sshd_config

## Actualizaciones de seguridad.

La mayoría de los incidentes de seguridad explotan vulnerabilidades conocidas para las que ya existe un parche. Mantener el sistema actualizado es la medida de seguridad con mejor relación esfuerzo/impacto.

En sistemas Debian/Ubuntu:

In [ ]:
# Lista paquetes con actualizaciones de seguridad disponibles
apt list --upgradable 2>/dev/null | grep -i security | head -10

El paquete ```unattended-upgrades``` permite automatizar la instalación de actualizaciones de seguridad:

```bash
sudo apt install unattended-upgrades
sudo dpkg-reconfigure unattended-upgrades
```

## Panorama de temas avanzados.

Los siguientes temas están fuera del alcance de este curso introductorio pero son parte esencial de la administración de seguridad en GNU/Linux.

### Control de acceso obligatorio (MAC).

Los permisos discrecionales vistos en este curso permiten al propietario de un recurso decidir quién accede a él. El control de acceso obligatorio agrega una capa de política que el sistema impone independientemente de los deseos del propietario.

* **SELinux** — desarrollado originalmente por la NSA, integrado en el kernel desde 2003. Predomina en el ecosistema Red Hat (RHEL, Fedora, CentOS).
* **AppArmor** — modelo más simple basado en rutas. Predomina en Debian y Ubuntu.

### Firewalls.

GNU/Linux filtra el tráfico de red mediante el subsistema *netfilter* del kernel. Las herramientas de espacio de usuario para administrarlo son:

* ```iptables``` / ```ip6tables``` — interfaz clásica, ampliamente documentada.
* ```nftables``` — reemplazo moderno de iptables, sintaxis unificada para IPv4/IPv6.
* ```firewalld``` / ```ufw``` — interfaces de alto nivel sobre nftables/iptables.

### Cifrado.

* **LUKS** — cifrado de discos y particiones a nivel de bloque.
* **GnuPG** — cifrado y firma de archivos y correo electrónico.
* **TLS/SSL** — cifrado de comunicaciones en red, gestionado típicamente mediante certificados.

### Auditoría avanzada.

* El subsistema ```auditd``` registra eventos del kernel con alta granularidad: llamadas al sistema, acceso a archivos, cambios de privilegios.
* Herramientas como ```lynis``` realizan auditorías automatizadas del sistema y generan recomendaciones de hardening.

---

Para un tratamiento completo de estos temas, consulte el **curso de seguridad en sistemas GNU/Linux** de Cloudevel.

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>